In [52]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

In [55]:
info_df = pd.read_csv("../data/input_info_VRC01_IC80.csv")
aligned_sequences = np.array([list(seq) for seq in info_df['Sequence']])
seq_no_array = info_df['Seq_no'].values
matches = aligned_sequences[:, None, :] == aligned_sequences[None, :, :]
seq_sim_mat = matches.mean(axis=2)

predict_df = pd.read_csv("../results/cv/predictions_summary_classification.csv")
pred_labels = []
for pred in predict_df["prediction"]:
    pred_0 = float(pred.split()[1].split(",")[0])
    pred_1 = float(pred.split()[3].split("}")[0])
    pred_labels.append(0 if pred_0 > pred_1 else 1)
predict_df["predicted_label"] = pred_labels

In [56]:
cutoff = 95

for n_fold in range(1,6):
    print("Processing fold -", n_fold)
    fold_df = predict_df[predict_df["fold"] == n_fold]
    print("-- Full dataset --")
    print("Accuracy:", accuracy_score(fold_df["actual_label"], fold_df["predicted_label"]))
    print("F1-score:", f1_score(fold_df["actual_label"], fold_df["predicted_label"]))
    
    train_fold_df = fold_df[fold_df["split"] == "Train"]
    train_fold_seqno = train_fold_df["Seq_no"].values
    train_idx = np.where(np.isin(seq_no_array, train_fold_seqno))[0]
    print(f"-- Train set (n={len(train_fold_seqno)}) --")
    print("Accuracy:", accuracy_score(train_fold_df["actual_label"], train_fold_df["predicted_label"]))
    print("F1-score:", f1_score(train_fold_df["actual_label"], train_fold_df["predicted_label"]))
    
    test_fold_df = fold_df[fold_df["split"] == "Test"]
    test_fold_seqno = test_fold_df["Seq_no"].values
    test_idx = np.where(np.isin(seq_no_array, test_fold_seqno))[0]
    print(f"-- Test set (n={len(test_fold_seqno)}) --")
    print("Accuracy:", accuracy_score(test_fold_df["actual_label"], test_fold_df["predicted_label"]))
    print("F1-score:", f1_score(test_fold_df["actual_label"], test_fold_df["predicted_label"]))

    sim_block = seq_sim_mat[np.ix_(test_idx, train_idx)] * 100
    n_test_in_train = (np.max(sim_block, axis=1) > cutoff).sum()
    overlap = n_test_in_train / len(test_idx)
    print(f"Train-test overlap (>{cutoff}% sim):", overlap, f"({n_test_in_train}/{len(test_idx)})")
    print("-------------------")

Processing fold - 1
-- Full dataset --
Accuracy: 0.8627671541057368
F1-score: 0.7910958904109588
-- Train set (n=711) --
Accuracy: 0.8945147679324894
F1-score: 0.8400852878464818
-- Test set (n=178) --
Accuracy: 0.7359550561797753
F1-score: 0.591304347826087
Train-test overlap (>95% sim): 0.398876404494382 (71/178)
-------------------
Processing fold - 2
-- Full dataset --
Accuracy: 0.8526434195725534
F1-score: 0.7805695142378559
-- Train set (n=711) --
Accuracy: 0.8762306610407876
F1-score: 0.8166666666666667
-- Test set (n=178) --
Accuracy: 0.7584269662921348
F1-score: 0.6324786324786325
Train-test overlap (>95% sim): 0.37640449438202245 (67/178)
-------------------
Processing fold - 3
-- Full dataset --
Accuracy: 0.8785151856017998
F1-score: 0.8235294117647058
-- Train set (n=711) --
Accuracy: 0.9184247538677919
F1-score: 0.8796680497925312
-- Test set (n=178) --
Accuracy: 0.7191011235955056
F1-score: 0.6153846153846154
Train-test overlap (>95% sim): 0.29213483146067415 (52/178)
---